In [34]:
from typing import Dict, List, Union

from pydantic import BaseModel
import requests

class URLParams(BaseModel):
    latitude: float
    longitude: float
    start_date: str
    end_date: str
    hourly: Union[str, List[str]]
    timezone: str


api_data_url="https://archive-api.open-meteo.com/v1/archive"

url_params_dict ={
    "latitude": 27,
    "longitude": 30,
    "start_date": "2025-04-29",
    "end_date": "2025-04-30",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "rain",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
    ],
    "timezone": "Africa/Cairo",
}

response = requests.get(api_data_url, params=url_params_dict)
if response.status_code == 200:
    data = response.json()
    print("success")

success


In [35]:
import pandas as pd


timestamp = data["hourly"]["time"]
temperature_2m = data["hourly"]["temperature_2m"]
relative_humidity_2m = data["hourly"]["relative_humidity_2m"]
rain = data["hourly"]["rain"]
precipitation = data["hourly"]["precipitation"]
cloud_cover = data["hourly"]["cloud_cover"]
wind_speed_10m = data["hourly"]["wind_speed_10m"]

# combine all features into a single dataframe
data = pd.DataFrame({
    "timestamp": timestamp,
    "temperature_2m": temperature_2m,
    "relative_humidity_2m": relative_humidity_2m,
    "rain": rain,
    "precipitation": precipitation,
    "cloud_cover": cloud_cover,
    "wind_speed_10m": wind_speed_10m
})

In [49]:
def split_data(df,train_size: float = 0.9):
    """
    Splits the data into training and testing sets.
    
    Args:
        df (pd.DataFrame): The DataFrame to split.
        train_size (float): The proportion of the data to include in the training set.
        
    Returns:
        tuple: A tuple containing the training and testing sets.
    """
    train_data = df[:int(len(df) * train_size)].copy()
    test_data = df[int(len(df) * train_size):].copy()
    return train_data, test_data

train, test = split_data(data, train_size=0.9)
def get_features_and_target(df: pd.DataFrame, target_col: str) -> (pd.DataFrame, pd.Series):
    """
    Splits the DataFrame into features and target variable.
    
    Args:
        df (pd.DataFrame): The DataFrame to split.
        target_col (str): The name of the target column.
        
    Returns:
        tuple: A tuple containing the features and target variable.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

target_col = "temperature_2m"
X_train, y_train = get_features_and_target(train, target_col)
X_test, y_test = get_features_and_target(test, target_col)

# convert to numpy arrays
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [52]:
from sktime.forecasting.fbprophet import Prophet

# Prepare the data for Prophet
# Prophet requires two columns: 'ds' for the datetime and 'y' for the target variable
prophet_data = data[['timestamp', 'temperature_2m']].rename(columns={'timestamp': 'ds', 'temperature_2m': 'y'})

# Initialize the Prophet model
model = Prophet()

# Fit the model
model.fit(prophet_data)

# Create a dataframe for future predictions
future = model.make_future_dataframe(periods=24, freq='H')  # Predict for the next 24 hours

# Make predictions
forecast = model.predict(future)

# Display the forecast
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

ModuleNotFoundError: Prophet requires package 'prophet' to be present in the python environment, but 'prophet' was not found. 'prophet' is a dependency of Prophet and required to construct it. To install the requirement 'prophet', please run: `pip install prophet` 